# LeetCode 1062 — Longest Repeating Substring bằng Python

Link đề: https://leetcode.com/problems/longest-repeating-substring/ · Độ khó: **Medium** · Chủ đề: String, Binary Search, Dynamic Programming, Suffix Array, Rolling Hash

> **Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết và theo bản mô tả trong kho tham khảo; nếu bạn có tài khoản Premium hãy **đối chiếu lại** đề gốc (đặc biệt về việc hai lần xuất hiện được phép **chồng lấn** hay không: ở bài này **được phép**, ví dụ `"aaaaa"` cho `4`).

Skeleton LeetCode:

```python
class Solution:
    def longestRepeatingSubstring(self, s: str) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**: độ dài lớn nhất `L` sao cho có một chuỗi con độ dài `L` xuất hiện **ít nhất hai lần** (ở hai vị trí bắt đầu khác nhau, có thể chồng lấn), hoặc `0` nếu không có. Điểm cốt lõi: tính chất **đơn điệu** (nếu độ dài `L` lặp thì `L - 1` cũng lặp) cho phép **tìm kiếm nhị phân theo độ dài**; hoặc DP "hậu tố chung dài nhất" `O(n²)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi con lặp (repeating substring) là chuỗi con xuất hiện ở **ít nhất hai vị trí bắt đầu khác nhau**; tìm độ dài lớn nhất.
2. Nắm công cụ: tập các chuỗi con (`set`), DP hậu tố chung `f[i][j]`, tìm kiếm nhị phân trên đáp án, băm cuộn (rolling hash), mảng hậu tố + LCP.
3. Đi từ **vét cạn theo định nghĩa** → **nhị phân + tập chuỗi con** → **DP `O(n²)`** → **nhị phân + băm cuộn** → **mảng hậu tố**.
4. Hiểu mấu chốt: `đáp án = max LCP của hai hậu tố khác nhau`, và hàm "có chuỗi con độ dài `L` lặp" **đơn điệu giảm** theo `L`.
5. Tránh lỗi kinh điển: cấm chồng lấn nhầm, cho phép `i == j` (đếm chính nó), bỏ sót cửa sổ cuối, băm không kiểm chứng (va chạm), nhị phân sai biên.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Trả về **độ dài** của chuỗi con lặp **dài nhất** (chuỗi con xuất hiện ít nhất hai lần trong `s`). Nếu không có chuỗi con lặp nào, trả về `0`.

**Ví dụ 1:** `s = "abcd"` → `0` (không có chuỗi con lặp).

**Ví dụ 2:** `s = "abbaba"` → `2` (`"ab"` và `"ba"` mỗi cái xuất hiện hai lần).

**Ví dụ 3:** `s = "aabcaabdaab"` → `3` (`"aab"` xuất hiện `3` lần).

**Ràng buộc:** `1 <= s.length <= 2000`; `s` chỉ gồm chữ cái thường.

Lưu ý: các lần xuất hiện **được phép chồng lấn** (ví dụ `"aaaa"` có `"aaa"` ở vị trí `0` và `1`, đáp án `3`).

In [1]:
import itertools
import os
import random
import time
from typing import List

examples = [("abcd", 0), ("abbaba", 2), ("aabcaabdaab", 3)]
for s, want in examples:
    print(f"s = {s!r} -> {want}")

s = 'abcd' -> 0
s = 'abbaba' -> 2
s = 'aabcaabdaab' -> 3


## 3. Kiến thức nền

### 3.1 Chuỗi con và số lần xuất hiện

Chuỗi con độ dài `L` bắt đầu tại `i` là `s[i:i+L]`. Số lần xuất hiện của `t` trong `s` là số vị trí `i` với `s[i:i+len(t)] == t` (đếm **cả** các lần chồng lấn). Chuỗi con "lặp" khi số lần xuất hiện `≥ 2`.

In [2]:
def occurrences(s, t):
    """Số vị trí bắt đầu i với s[i:i+len(t)] == t (tính cả các lần chồng lấn)."""
    return sum(1 for i in range(len(s) - len(t) + 1) if s[i:i + len(t)] == t)


print(occurrences("aaaa", "aaa"), occurrences("abbaba", "ab"), occurrences("abbaba", "ba"), occurrences("abcd", "a"))
assert occurrences("aaaa", "aaa") == 2 and occurrences("abbaba", "ab") == 2

2 2 2 1


### 3.2 Tính đơn điệu

Nếu chuỗi con `t` độ dài `L` xuất hiện tại hai vị trí `i ≠ j` thì tiền tố độ dài `L - 1` của `t` cũng xuất hiện tại `i` và `j`. Vậy tập các `L` "có chuỗi con lặp" là một **đoạn đầu** `{0, 1, …, ans}`: hàm `has_repeat(L)` đơn điệu giảm (đúng rồi sai). Đây là điều kiện để **tìm kiếm nhị phân trên đáp án**.

### 3.3 Hậu tố chung dài nhất

Đặt `f[i][j]` (với `j < i`) là độ dài **hậu tố chung dài nhất** của `s[:i+1]` và `s[:j+1]`, tức số `m` lớn nhất sao cho `s[i-m+1..i] == s[j-m+1..j]`. Nếu `s[i] != s[j]` thì `f[i][j] = 0`; nếu bằng thì `f[i][j] = f[i-1][j-1] + 1`. Đáp án là `max f[i][j]` trên mọi cặp `j < i`: hai chuỗi con cùng độ dài kết thúc tại `i` và `j` khác nhau tương ứng hai vị trí bắt đầu khác nhau. Chồng lấn hoàn toàn được phép.

## 4. Ý tưởng nền tảng / trực giác

Một chuỗi con lặp tức là có hai "con trỏ" `j < i` mà các ký tự **kết thúc** tại hai vị trí đó khớp nhau lùi về trước càng dài càng tốt. Chạy từng cặp `(i, j)` và kéo dài khớp: DP. Hoặc nhìn theo độ dài: "có độ dài `L` nào lặp không?" là câu hỏi đúng/sai đơn điệu, đoán `L` bằng nhị phân và kiểm tra bằng tập các cửa sổ độ dài `L`.

## 5. Brute force — thử mọi chuỗi con theo định nghĩa

**Ý tưởng:** với độ dài `L` từ lớn xuống nhỏ, duyệt mọi chuỗi con `t` độ dài `L` và **đếm số lần xuất hiện bằng định nghĩa** (`occurrences`); gặp `t` có `≥ 2` lần thì trả `L`. Đúng theo định nghĩa, làm **oracle**; chậm (`O(n⁴)`), dùng cho `n ≤ 14`.

In [3]:
def lrs_brute(s):
    n = len(s)
    for L in range(n - 1, 0, -1):
        for i in range(n - L + 1):
            if occurrences(s, s[i:i + L]) >= 2:
                return L
    return 0


for s, want in examples:
    assert lrs_brute(s) == want
print([lrs_brute(s) for s, _ in examples])

[0, 2, 3]


In [4]:
# Code nộp LeetCode cho Brute force (đúng nhưng O(n^4): sẽ TLE khi n lớn)
class SolutionBrute:
    def longestRepeatingSubstring(self, s: str) -> int:
        n = len(s)
        for L in range(n - 1, 0, -1):
            for i in range(n - L + 1):
                t = s[i:i + L]
                cnt = 0
                for j in range(n - L + 1):
                    if s[j:j + L] == t:
                        cnt += 1
                if cnt >= 2:
                    return L
        return 0

**Độ phức tạp:** `O(n⁴)` thời gian (`n` độ dài × `n` chuỗi con × `n` vị trí × `n` so sánh), `O(n)` bộ nhớ.

**Nút thắt:** đếm lại từ đầu cho mỗi chuỗi con; chưa tận dụng đơn điệu.

## 6. Bước đột phá

Hai hướng: (i) đơn điệu ⇒ **nhị phân theo `L`**, kiểm tra bằng một tập băm các cửa sổ độ dài `L`: `O(n² log n)` hoặc `O(n log n)` với băm cuộn; (ii) DP hậu tố chung `O(n²)` không cần nhị phân.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** nhị phân + tập chuỗi con, **Cách 3** DP hậu tố chung (đáp án chính), **Cách 4** nhị phân + băm cuộn, **Cách 5** mảng hậu tố + LCP.

### Cách 2 — Nhị phân theo độ dài + tập chuỗi con

**Giải thích:** `has_repeat(L)` = có hai cửa sổ độ dài `L` bằng nhau (dùng `set`). Nhị phân `L ∈ [0, n-1]` để tìm `L` lớn nhất thoả (vì đơn điệu).

In [5]:
def has_repeat_set(s, L):
    seen = set()
    for i in range(len(s) - L + 1):
        t = s[i:i + L]
        if t in seen:
            return True
        seen.add(t)
    return False


def lrs_bs_set(s):
    lo, hi = 0, len(s) - 1                            # tìm L lớn nhất với has_repeat(L); has_repeat(0) coi là đúng
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if has_repeat_set(s, mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


for s, want in examples:
    assert lrs_bs_set(s) == want
print([lrs_bs_set(s) for s, _ in examples])

[0, 2, 3]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionBSSet:
    def longestRepeatingSubstring(self, s: str) -> int:
        def ok(L):
            seen = set()
            for i in range(len(s) - L + 1):
                t = s[i:i + L]
                if t in seen:
                    return True
                seen.add(t)
            return False

        lo, hi = 0, len(s) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

**Độ phức tạp:** `O(n² log n)` thời gian (mỗi lần kiểm tra tạo `n` chuỗi con độ dài `L`), `O(n²)` bộ nhớ trong trường hợp xấu.

### Cách 3 — DP hậu tố chung (đáp án chính)

**Ý tưởng:** `f[i][j] = f[i-1][j-1] + 1` nếu `s[i] == s[j]`, ngược lại `0`, với `j < i`. Chỉ cần hàng trước nên dùng hai mảng `O(n)`. Đáp án là giá trị lớn nhất gặp được.

In [7]:
def lrs_dp(s):
    n = len(s)
    best = 0
    prev = [0] * (n + 1)                              # prev[j + 1] = f[i - 1][j]
    for i in range(n):
        cur = [0] * (n + 1)
        for j in range(i):
            if s[i] == s[j]:
                cur[j + 1] = prev[j] + 1              # f[i][j] = f[i-1][j-1] + 1 (prev[0] = 0 khi j = 0)
                if cur[j + 1] > best:
                    best = cur[j + 1]
        prev = cur
    return best


for s, want in examples:
    assert lrs_dp(s) == want
print([lrs_dp(s) for s, _ in examples])

[0, 2, 3]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionDP:
    def longestRepeatingSubstring(self, s: str) -> int:
        n = len(s)
        best = 0
        prev = [0] * (n + 1)
        for i in range(n):
            cur = [0] * (n + 1)
            for j in range(i):
                if s[i] == s[j]:
                    cur[j + 1] = prev[j] + 1
                    if cur[j + 1] > best:
                        best = cur[j + 1]
            prev = cur
        return best

**Độ phức tạp:** `O(n²)` thời gian (`≈ n²/2 = 2·10^6` bước khi `n = 2000`), `O(n)` bộ nhớ.

### Cách 4 — Nhị phân + băm cuộn (rolling hash)

**Ý tưởng:** thay `set` các chuỗi con bằng `set`/`dict` các **giá trị băm** tính cuộn trong `O(1)` mỗi cửa sổ: tổng `O(n log n)` kỳ vọng. Để tránh sai do **va chạm** băm, khi hai cửa sổ cùng băm ta **so sánh trực tiếp** hai chuỗi con (nếu khác thì tiếp tục).

In [9]:
MOD = (1 << 61) - 1
BASE = 911382323


def has_repeat_hash(s, L, mod=MOD, base=BASE, verify=True):
    n = len(s)
    if L == 0:
        return True
    if L > n - 1:
        return False
    h = 0
    for ch in s[:L]:
        h = (h * base + ord(ch)) % mod
    power = pow(base, L, mod)
    seen = {h: [0]}
    for i in range(1, n - L + 1):
        h = (h * base + ord(s[i + L - 1]) - ord(s[i - 1]) * power) % mod     # trượt cửa sổ
        if h in seen:
            if not verify:
                return True
            if any(s[j:j + L] == s[i:i + L] for j in seen[h]):
                return True
            seen[h].append(i)
        else:
            seen[h] = [i]
    return False


def lrs_bs_hash(s):
    lo, hi = 0, len(s) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if has_repeat_hash(s, mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


for s, want in examples:
    assert lrs_bs_hash(s) == want
print([lrs_bs_hash(s) for s, _ in examples])

[0, 2, 3]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionHash:
    def longestRepeatingSubstring(self, s: str) -> int:
        n = len(s)
        MOD, BASE = (1 << 61) - 1, 911382323

        def ok(L):
            if L == 0:
                return True
            h = 0
            for ch in s[:L]:
                h = (h * BASE + ord(ch)) % MOD
            power = pow(BASE, L, MOD)
            seen = {h: [0]}
            for i in range(1, n - L + 1):
                h = (h * BASE + ord(s[i + L - 1]) - ord(s[i - 1]) * power) % MOD
                if h in seen:
                    if any(s[j:j + L] == s[i:i + L] for j in seen[h]):
                        return True
                    seen[h].append(i)
                else:
                    seen[h] = [i]
            return False

        lo, hi = 0, n - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

**Độ phức tạp:** `O(n log n)` kỳ vọng (với kiểm chứng; trường hợp xấu có thể cao hơn nếu rất nhiều va chạm thật), `O(n)` bộ nhớ.

### Cách 5 — Mảng hậu tố + LCP kề nhau

**Ý tưởng:** sắp xếp các hậu tố `s[i:]` theo thứ tự từ điển; hai hậu tố có tiền tố chung dài nhất luôn **kề nhau** trong thứ tự này, nên đáp án `= max` độ dài tiền tố chung của mọi cặp hậu tố kề nhau (LCP).

In [11]:
def lrs_suffix_array(s):
    n = len(s)
    sa = sorted(range(n), key=lambda i: s[i:])
    best = 0
    for a, b in zip(sa, sa[1:]):
        best = max(best, len(os.path.commonprefix([s[a:], s[b:]])))
    return best


for s, want in examples:
    assert lrs_suffix_array(s) == want
print([lrs_suffix_array(s) for s, _ in examples])

[0, 2, 3]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionSuffixArray:
    def longestRepeatingSubstring(self, s: str) -> int:
        n = len(s)
        sa = sorted(range(n), key=lambda i: s[i:])
        best = 0
        for a, b in zip(sa, sa[1:]):
            k = 0
            while a + k < n and b + k < n and s[a + k] == s[b + k]:
                k += 1
            best = max(best, k)
        return best

**Độ phức tạp:** sắp xếp `O(n² log n)` (so sánh hậu tố tốn `O(n)`), LCP `O(n²)` trường hợp xấu; `O(n²)` bộ nhớ cho các khoá. Với `n ≤ 2000` vẫn chạy được.

## 8. TRACE từng bước

Trace **Cách 3** với `s = "abbaba"`. Chỉ in các cặp `(i, j)`, `j < i`, có `s[i] == s[j]`:

| `i` | `j` | `s[i] = s[j]` | `f[i][j] = f[i-1][j-1] + 1` |
|---|---|---|---|
| 2 (`b`) | 1 | `b` | `f[1][0] + 1 = 0 + 1 = 1` |
| 3 (`a`) | 0 | `a` | `1` (j = 0) |
| 4 (`b`) | 1 | `b` | `f[3][0] + 1 = 2` (`"ab"` kết thúc tại 4 và 1) |
| 4 (`b`) | 2 | `b` | `f[3][1] + 1 = 0 + 1 = 1` |
| 5 (`a`) | 0 | `a` | `1` |
| 5 (`a`) | 3 | `a` | `f[4][2] + 1 = 2` (`"ba"` kết thúc tại 5 và 3) |

Lớn nhất `= 2`. Hàm trace in từng bước:

In [13]:
def trace_dp(s):
    n = len(s)
    f = [[0] * n for _ in range(n)]
    best = 0
    print(f"s = {s!r}")
    for i in range(n):
        for j in range(i):
            if s[i] == s[j]:
                f[i][j] = (f[i - 1][j - 1] if j > 0 else 0) + 1
                best = max(best, f[i][j])
                print(f"  i = {i} ({s[i]!r}), j = {j}: f = {f[i][j]}  (chuỗi con {s[i - f[i][j] + 1:i + 1]!r})")
    print("  đáp án =", best)
    return best


assert trace_dp("abbaba") == 2
assert trace_dp("aabcaabdaab") == 3

s = 'abbaba'
  i = 2 ('b'), j = 1: f = 1  (chuỗi con 'b')
  i = 3 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 4 ('b'), j = 1: f = 2  (chuỗi con 'ab')
  i = 4 ('b'), j = 2: f = 1  (chuỗi con 'b')
  i = 5 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 5 ('a'), j = 3: f = 2  (chuỗi con 'ba')
  đáp án = 2
s = 'aabcaabdaab'
  i = 1 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 4 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 4 ('a'), j = 1: f = 1  (chuỗi con 'a')
  i = 5 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 5 ('a'), j = 1: f = 2  (chuỗi con 'aa')
  i = 5 ('a'), j = 4: f = 1  (chuỗi con 'a')
  i = 6 ('b'), j = 2: f = 3  (chuỗi con 'aab')
  i = 8 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 8 ('a'), j = 1: f = 1  (chuỗi con 'a')
  i = 8 ('a'), j = 4: f = 1  (chuỗi con 'a')
  i = 8 ('a'), j = 5: f = 1  (chuỗi con 'a')
  i = 9 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 9 ('a'), j = 1: f = 2  (chuỗi con 'aa')
  i = 9 ('a'), j = 4: f = 1  (chuỗi con 'a')
  i = 9 ('a'), j = 5: f = 2  (chuỗi con 'aa')
  i 

**Trace nhị phân (Cách 2)** với `s = "abbaba"`: `lo = 0, hi = 5` → `mid = 3`: các cửa sổ `abb, bba, aba, bab`, không trùng ⇒ `hi = 2`; `mid = 1`: có `a` lặp ⇒ `lo = 1`; `mid = 2`: `ab, bb, ba, ab, ba` ⇒ có `ab` lặp ⇒ `lo = 2`; `lo = hi = 2`.

In [14]:
def trace_bs(s):
    lo, hi = 0, len(s) - 1
    print(f"s = {s!r}")
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok = has_repeat_set(s, mid)
        print(f"  lo = {lo}, hi = {hi}, mid = {mid}: có lặp = {ok}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    print("  đáp án =", lo)
    return lo


assert trace_bs("abbaba") == 2

s = 'abbaba'
  lo = 0, hi = 5, mid = 3: có lặp = False
  lo = 0, hi = 2, mid = 1: có lặp = True
  lo = 1, hi = 2, mid = 2: có lặp = True
  đáp án = 2


## 9. Code nộp LeetCode

Chọn **Cách 3** (DP hậu tố chung, `O(n²)` thời gian, `O(n)` bộ nhớ). Giữ nguyên `class Solution`, `longestRepeatingSubstring(self, s)`:

In [15]:
class Solution:
    def longestRepeatingSubstring(self, s: str) -> int:
        n = len(s)
        best = 0
        prev = [0] * (n + 1)                        # prev[j + 1] = f[i-1][j]
        for i in range(n):
            cur = [0] * (n + 1)
            for j in range(i):                      # j < i: hai vị trí kết thúc khác nhau
                if s[i] == s[j]:
                    cur[j + 1] = prev[j] + 1        # kéo dài hậu tố chung thêm 1
                    if cur[j + 1] > best:
                        best = cur[j + 1]
            prev = cur
        return best


print([Solution().longestRepeatingSubstring(s) for s, _ in examples])
assert all(Solution().longestRepeatingSubstring(s) == v for s, v in examples)

[0, 2, 3]


## 10. Vì sao đúng

**(a) Đáp án = max hậu tố chung.** Một chuỗi con `t` độ dài `L` lặp ⇔ tồn tại `j < i` với `s[i-L+1..i] == s[j-L+1..j]`, tức hậu tố chung của `s[:i+1]` và `s[:j+1]` có độ dài `≥ L`. Do đó đáp án `= max_{j<i} lcsuffix(i, j)`. (Hai vị trí kết thúc khác nhau ⇔ hai vị trí bắt đầu khác nhau, vì cùng độ dài.)

**(b) Công thức truy hồi.** Nếu `s[i] != s[j]` thì hậu tố chung `= 0`. Nếu bằng nhau, hậu tố chung dài nhất của `(i, j)` là hậu tố chung dài nhất của `(i-1, j-1)` cộng thêm ký tự cuối chung (tính cả khi `j = 0`: `f[i-1][-1] = 0`). Chú ý `(i-1, j-1)` vẫn thoả `j-1 < i-1`, không cần điều kiện không chồng lấn nên chồng lấn được tính đúng.

**(c) Hàng cuộn.** `f[i][·]` chỉ phụ thuộc `f[i-1][·]`; ta chỉ lưu hàng trước. Chỉ số dịch `+1` để tránh `-1` ở `j = 0`.

**(d) Đơn điệu (cho nhị phân).** Như mục 3.2: tiền tố của chuỗi con lặp cũng lặp tại cùng hai vị trí. Nhị phân tìm `L` lớn nhất có `has_repeat(L)`; `L = 0` coi là đúng, `L = n` luôn sai (chỉ một cửa sổ), nên `hi = n - 1`.

**(e) Băm.** Băm cuộn chỉ cho "ứng viên"; kiểm chứng bằng so sánh chuỗi trực tiếp loại hoàn toàn dương tính giả; không có âm tính giả vì hai chuỗi bằng nhau có băm bằng nhau.

**(f) Mảng hậu tố.** Với ba hậu tố `A ≤ B ≤ C` theo thứ tự từ điển, `lcp(A, C) = min(lcp(A, B), lcp(B, C))`; do đó cặp có LCP lớn nhất luôn kề nhau.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp vét cạn theo định nghĩa trên mọi chuỗi nhị phân `n ≤ 12`, mọi chuỗi `{a,b,c}` `n ≤ 7` và các chuỗi ngẫu nhiên (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(n⁴)` | `O(n)` | oracle `n ≤ 14` |
| 2. Nhị phân + `set` | `O(n² log n)` | `O(n²)` | đơn giản |
| 3. DP hậu tố chung | `O(n²)` | `O(n)` | **đáp án chính** |
| 4. Nhị phân + băm cuộn | `O(n log n)` kỳ vọng | `O(n)` | nhanh nhất khi `n` lớn |
| 5. Mảng hậu tố + LCP | `O(n² log n)` | `O(n²)` | nền tảng cho bài "chuỗi con lặp" tổng quát |

Đo thực tế với `n = 2000`:

In [16]:
def bench(fn, arg, repeat=2):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1062)
cases = {
    "ngẫu nhiên 26 chữ": "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(2000)),
    "ngẫu nhiên {a,b}": "".join(rng.choice("ab") for _ in range(2000)),
    "toàn a": "a" * 2000,
}
for label, s in cases.items():
    print(label)
    for name, fn in [("bs_set", lrs_bs_set), ("dp", lrs_dp), ("bs_hash", lrs_bs_hash), ("suffix_array", lrs_suffix_array)]:
        print(f"  {name:13s}: {bench(fn, s) * 1000:8.1f} ms -> {fn(s)}")

ngẫu nhiên 26 chữ
  bs_set       :      5.3 ms -> 4


  dp           :    105.8 ms -> 4
  bs_hash      :     10.2 ms -> 4
  suffix_array :      3.1 ms -> 4
ngẫu nhiên {a,b}
  bs_set       :      3.4 ms -> 20


  dp           :    134.2 ms -> 20
  bs_hash      :      6.9 ms -> 20
  suffix_array :      3.9 ms -> 20
toàn a
  bs_set       :      0.0 ms -> 1999


  dp           :    168.2 ms -> 1999
  bs_hash      :      2.5 ms -> 1999
  suffix_array :     64.5 ms -> 1999


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "bs_set": lrs_bs_set,
    "dp": lrs_dp,
    "bs_hash": lrs_bs_hash,
    "suffix_array": lrs_suffix_array,
    "Solution": lambda s: Solution().longestRepeatingSubstring(s),
    "SolutionBSSet": lambda s: SolutionBSSet().longestRepeatingSubstring(s),
    "SolutionDP": lambda s: SolutionDP().longestRepeatingSubstring(s),
    "SolutionHash": lambda s: SolutionHash().longestRepeatingSubstring(s),
    "SolutionSuffixArray": lambda s: SolutionSuffixArray().longestRepeatingSubstring(s),
}
BRUTE = {"brute": lrs_brute, "SolutionBrute": lambda s: SolutionBrute().longestRepeatingSubstring(s)}


def check(s, want=None, brute=False):
    if want is None:
        want = lrs_brute(s) if brute else Solution().longestRepeatingSubstring(s)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(s)
        assert got == want, (name, s, got, want)
    return want


for s, want in examples:
    check(s, want, brute=True)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn


### B. Edge case

- `n = 1` (`"a"`): `0`; `n = 2`: `"aa"` → `1`, `"ab"` → `0`.
- Toàn một ký tự: `"aaaa"` → `3` (chồng lấn được phép), `"a" * n → n - 1`.
- Mọi ký tự khác nhau → `0`.
- Chỉ lặp một ký tự (`"abca"` → `1`).
- Lặp ở đầu và cuối (`"abxab"` → `2`); lặp chồng lấn (`"ababa"` → `3`: `"aba"` ở `0` và `2`).
- Lặp ngay cạnh nhau (`"abab"` → `2`; `"abcabc"` → `3`).
- Chuỗi tuần hoàn dài.

In [18]:
edge = [
    ("a", 0), ("aa", 1), ("ab", 0), ("aaaa", 3), ("aaaaa", 4), ("abcdefg", 0), ("abca", 1), ("abxab", 2),
    ("ababa", 3), ("abab", 2), ("abcabc", 3), ("abcabcabc", 6), ("aabaab", 3), ("ab" * 6, 10), ("z" * 12, 11),
    ("abcdefghijklmnopqrstuvwxyz", 0), ("abcdefghijklmnopqrstuvwxyza", 1), ("banana", 3), ("mississippi", 4),
]
for s, want in edge:
    got = check(s, want, brute=len(s) <= 14)
    print(f"  {s!r:30} -> {got}")
print(len(edge), "edge case: mọi cài đặt khớp")

  'a'                            -> 0
  'aa'                           -> 1
  'ab'                           -> 0
  'aaaa'                         -> 3
  'aaaaa'                        -> 4
  'abcdefg'                      -> 0
  'abca'                         -> 1
  'abxab'                        -> 2
  'ababa'                        -> 3
  'abab'                         -> 2
  'abcabc'                       -> 3
  'abcabcabc'                    -> 6
  'aabaab'                       -> 3
  'abababababab'                 -> 10
  'zzzzzzzzzzzz'                 -> 11
  'abcdefghijklmnopqrstuvwxyz'   -> 0
  'abcdefghijklmnopqrstuvwxyza'  -> 1
  'banana'                       -> 3
  'mississippi'                  -> 4
19 edge case: mọi cài đặt khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_non_overlapping(s):                              # cấm chồng lấn: giới hạn hậu tố chung <= i - j
    n = len(s)
    best = 0
    prev = [0] * (n + 1)
    for i in range(n):
        cur = [0] * (n + 1)
        for j in range(i):
            if s[i] == s[j]:
                cur[j + 1] = min(prev[j] + 1, i - j)
                best = max(best, cur[j + 1])
        prev = cur
    return best


def bug_include_self(s):                                 # cho phép j == i: chuỗi con "lặp" với chính nó
    n = len(s)
    best = 0
    prev = [0] * (n + 1)
    for i in range(n):
        cur = [0] * (n + 1)
        for j in range(i + 1):
            if s[i] == s[j]:
                cur[j + 1] = prev[j] + 1
                best = max(best, cur[j + 1])
        prev = cur
    return best


def bug_char_repeat_only(s):                             # chỉ xét ký tự đơn
    return 1 if len(set(s)) < len(s) else 0


def bug_missing_last_window(s):                          # nhị phân với kiểm tra thiếu cửa sổ cuối
    def ok(L):
        seen = set()
        for i in range(len(s) - L):                      # lẽ ra range(len(s) - L + 1)
            t = s[i:i + L]
            if t in seen:
                return True
            seen.add(t)
        return False
    lo, hi = 0, len(s) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


def bug_hash_no_verify(s):                               # băm modulo nhỏ, không kiểm chứng
    lo, hi = 0, len(s) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if has_repeat_hash(s, mid, mod=7, base=3, verify=False):
            lo = mid
        else:
            hi = mid - 1
    return lo


def bug_prefix_only(s):                                  # chỉ so sánh với hậu tố bắt đầu từ vị trí 0
    best = 0
    for i in range(1, len(s)):
        k = 0
        while i + k < len(s) and s[k] == s[i + k]:
            k += 1
        best = max(best, k)
    return best


BUGS = [bug_non_overlapping, bug_include_self, bug_char_repeat_only, bug_missing_last_window, bug_hash_no_verify, bug_prefix_only]
pool = ["".join(t) for n in range(1, 9) for t in itertools.product("ab", repeat=n)]
for bug in BUGS:
    found = next((s for s in pool if bug(s) != lrs_brute(s)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:25s} sai trên {found!r}: ra {bug(found)}, đúng {lrs_brute(found)}")

bug_non_overlapping       sai trên 'aaa': ra 1, đúng 2
bug_include_self          sai trên 'a': ra 1, đúng 0
bug_char_repeat_only      sai trên 'aaa': ra 1, đúng 2
bug_missing_last_window   sai trên 'aa': ra 0, đúng 1
bug_hash_no_verify        sai trên 'abab': ra 3, đúng 2
bug_prefix_only           sai trên 'abb': ra 0, đúng 1


### D. Vét cạn mọi chuỗi nhỏ

Mọi chuỗi nhị phân độ dài `1..12` (`8 190` chuỗi) và mọi chuỗi `{a,b,c}` độ dài `1..7` (`3 279` chuỗi): so mọi cài đặt với vét cạn theo định nghĩa.

In [20]:
t0 = time.perf_counter()
cnt = 0
for alpha, nmax in (("ab", 12), ("abc", 7)):
    for n in range(1, nmax + 1):
        for tup in itertools.product(alpha, repeat=n):
            s = "".join(tup)
            want = lrs_brute(s)
            for name, fn in IMPLS.items():
                assert fn(s) == want, (name, s)
            cnt += 1
print(cnt, "chuỗi:", len(IMPLS), "cài đặt khớp vét cạn theo định nghĩa;", round(time.perf_counter() - t0, 1), "s")

11469 chuỗi: 9 cài đặt khớp vét cạn theo định nghĩa; 1.3 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Chuỗi ngẫu nhiên `n ≤ 14`: khớp vét cạn; (2) `n` lớn: mọi cách khớp nhau; (3) **đảo chuỗi** không đổi đáp án; (4) **hoán vị bảng chữ cái** không đổi đáp án; (5) **`s + s`** có đáp án `≥ len(s)`; (6) **chuỗi gài sẵn**: chèn cùng một chuỗi con độ dài `m` hai lần (có thể chồng lấn) ⇒ đáp án `≥ m`; (7) bất biến: `ans = n - 1 ⇔ s` gồm một ký tự; `ans ≥ 1 ⇔` có ký tự lặp; `ans ≤ n - 1`.

In [21]:
rng = random.Random(20240162)
t0 = time.perf_counter()
for it in range(400):
    n = rng.randint(1, 14)
    alpha = rng.choice(["ab", "abc", "abcd", "abcdefgh"])
    s = "".join(rng.choice(alpha) for _ in range(n))
    check(s, brute=True)                                                              # (1)
print("(1) 400 chuỗi ngẫu nhiên n <= 14 khớp vét cạn:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(200):
    n = rng.randint(1, 300)
    alpha = rng.choice(["ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alpha) for _ in range(n))
    ans = check(s)                                                                    # (2)
    assert Solution().longestRepeatingSubstring(s[::-1]) == ans                       # (3)
    perm = list(alpha)
    rng.shuffle(perm)
    tr = str.maketrans(alpha, "".join(perm))
    assert Solution().longestRepeatingSubstring(s.translate(tr)) == ans              # (4)
    assert Solution().longestRepeatingSubstring(s + s) >= n                           # (5)
    assert (ans == n - 1) == (len(set(s)) == 1 and n >= 2) or n == 1                  # (7)
    assert (ans >= 1) == (len(set(s)) < n) and ans <= max(n - 1, 0)
    m = rng.randint(1, 10)
    t = "".join(rng.choice("xyz") for _ in range(m))
    gap = rng.randint(0, 30)
    planted = "".join(rng.choice("abc") for _ in range(rng.randint(0, 20))) + t + "".join(rng.choice("abc") for _ in range(gap)) + t
    assert Solution().longestRepeatingSubstring(planted) >= m                          # (6)
    assert Solution().longestRepeatingSubstring("q" + t + "r" + t) >= m
print("(2)-(7) 200 chuỗi n <= 300 + chuỗi gài sẵn:", round(time.perf_counter() - t0, 1), "s")

(1) 400 chuỗi ngẫu nhiên n <= 14 khớp vét cạn: 0.1 s


(2)-(7) 200 chuỗi n <= 300 + chuỗi gài sẵn: 2.1 s


### F. Giới hạn

`n = 2000`: toàn `a` (đáp án `1999`), chuỗi tuần hoàn `ab`, hai nửa giống nhau (đáp án `≥ 1000`), chuỗi ngẫu nhiên 26 chữ, toàn khác nhau trên alphabet nhỏ (de Bruijn-like tăng dần), `a…ab…b`. Các cách nhanh phải khớp.

In [22]:
t0 = time.perf_counter()
rg = random.Random(5)
half = "".join(rg.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(1000))
cases = {
    "toàn a": ("a" * 2000, 1999),
    "ab tuần hoàn": ("ab" * 1000, 1998),
    "hai nửa giống nhau": (half + half, None),
    "ngẫu nhiên 26 chữ": ("".join(rg.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(2000)), None),
    "a..a b..b": ("a" * 1000 + "b" * 1000, 999),
    "ngẫu nhiên {a,b}": ("".join(rg.choice("ab") for _ in range(2000)), None),
}
for name, (s, want) in cases.items():
    got = Solution().longestRepeatingSubstring(s)
    for iname, fn in IMPLS.items():
        if iname.startswith("Solution") and iname != "Solution":
            continue
        assert fn(s) == got, (iname, name)
    if want is not None:
        assert got == want, (name, got, want)
    print(f"  {name:20s}: n = {len(s)}, đáp án = {got}")
assert cases["hai nửa giống nhau"][0][:1000] == cases["hai nửa giống nhau"][0][1000:]
assert Solution().longestRepeatingSubstring(cases["hai nửa giống nhau"][0]) >= 1000
print("n = 2000:", round(time.perf_counter() - t0, 1), "s")

  toàn a              : n = 2000, đáp án = 1999


  ab tuần hoàn        : n = 2000, đáp án = 1998


  hai nửa giống nhau  : n = 2000, đáp án = 1000


  ngẫu nhiên 26 chữ   : n = 2000, đáp án = 4


  a..a b..b           : n = 2000, đáp án = 999


  ngẫu nhiên {a,b}    : n = 2000, đáp án = 20
n = 2000: 2.6 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cấm chồng lấn

In [23]:
print("bug:", bug_non_overlapping("aaaaa"), "| đúng:", Solution().longestRepeatingSubstring("aaaaa"))
assert bug_non_overlapping("aaaaa") == 2 and Solution().longestRepeatingSubstring("aaaaa") == 4

bug: 2 | đúng: 4


**Vì sao sai:** đề chỉ cần chuỗi con xuất hiện ở **hai vị trí bắt đầu khác nhau**, không đòi hỏi hai lần không giao nhau. `"aaaaa"` có `"aaaa"` ở vị trí `0` và `1`. (Biến thể "không chồng lấn" là bài khác, xem mục 15.)

### Lỗi 2: cho phép `j == i`

In [24]:
print("bug:", bug_include_self("abcd"), "| đúng:", Solution().longestRepeatingSubstring("abcd"))
assert bug_include_self("abcd") == 4 and Solution().longestRepeatingSubstring("abcd") == 0

bug: 4 | đúng: 0


**Vì sao sai:** nếu `j = i` thì mọi chuỗi con "khớp" với chính nó. Phải `j < i` (hai vị trí khác nhau).

### Lỗi 3: chỉ xét ký tự đơn

In [25]:
print("bug:", bug_char_repeat_only("abbaba"), "| đúng:", Solution().longestRepeatingSubstring("abbaba"))
assert bug_char_repeat_only("abbaba") == 1 and Solution().longestRepeatingSubstring("abbaba") == 2

bug: 1 | đúng: 2


**Vì sao sai:** chuỗi con lặp có thể dài hơn `1`; phải kéo dài khớp (DP) hoặc thử các độ dài.

### Lỗi 4: kiểm tra thiếu cửa sổ cuối

In [26]:
bad = next(s for s in pool if bug_missing_last_window(s) != lrs_brute(s))
print("phản ví dụ:", repr(bad), "| bug:", bug_missing_last_window(bad), "| đúng:", lrs_brute(bad))
assert bug_missing_last_window(bad) != lrs_brute(bad)

phản ví dụ: 'aa' | bug: 0 | đúng: 1


**Vì sao sai:** có `n - L + 1` cửa sổ độ dài `L` (vị trí bắt đầu `0..n-L`). `range(n - L)` bỏ mất cửa sổ cuối.

### Lỗi 5: băm không kiểm chứng

In [27]:
bad = next(s for s in pool if bug_hash_no_verify(s) != lrs_brute(s))
print("phản ví dụ:", repr(bad), "| bug:", bug_hash_no_verify(bad), "| đúng:", lrs_brute(bad))
assert bug_hash_no_verify(bad) != lrs_brute(bad) and Solution().longestRepeatingSubstring(bad) == lrs_brute(bad)

phản ví dụ: 'abab' | bug: 3 | đúng: 2


**Vì sao sai:** băm trùng không đảm bảo chuỗi trùng (va chạm). Ở đây modulo nhỏ làm va chạm xảy ra thường xuyên; với modulo lớn xác suất nhỏ nhưng **vẫn nên kiểm chứng** hoặc dùng hai modulo.

### Lỗi 6 (bonus): chỉ so với hậu tố bắt đầu ở `0`

In [28]:
bad = next(s for s in pool if bug_prefix_only(s) != lrs_brute(s))
print("phản ví dụ:", repr(bad), "| bug:", bug_prefix_only(bad), "| đúng:", lrs_brute(bad))
assert bug_prefix_only(bad) < lrs_brute(bad)

phản ví dụ: 'abb' | bug: 0 | đúng: 1


**Vì sao sai:** hai lần xuất hiện có thể ở bất kỳ hai vị trí, không nhất thiết một lần bắt đầu tại `0`. Phải xét mọi cặp.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chuỗi con lặp dài nhất / dài nhất xuất hiện ≥ 2 lần" → **nhị phân theo độ dài** (đơn điệu) + tập chuỗi con/băm; hoặc **DP hậu tố chung `O(n²)`** khi `n ≤ 2000`; hoặc **mảng hậu tố + LCP** khi cần nhiều truy vấn.

**Mẹo:**

1. Nhị phân trên đáp án khi có đơn điệu: "có độ dài `L` thoả" đúng rồi sai.
2. DP `f[i][j] = f[i-1][j-1] + 1` chỉ cần hàng trước; dịch chỉ số `+1` tránh biên.
3. Băm cuộn: luôn trừ ký tự ra khỏi cửa sổ với `power = base^L`; kiểm chứng khi trùng.
4. Hai vị trí khác nhau: `j < i`.
5. Oracle: đếm lần xuất hiện **theo định nghĩa** (`occurrences`) cho `n ≤ 14`.
6. Nhớ chồng lấn được phép ở bài này (bài "không chồng lấn" cần ràng buộc `i - j ≥ L`).

**Họ bài cùng pattern:** Longest Duplicate Substring (1044), Longest Common Subsequence (1143), Maximum Length of Repeated Subarray (718), Distinct Echo Substrings (1316), Repeated Substring Pattern (459), Longest Happy Prefix (1392).

## 15. Biến thể

1. **Trả về chính chuỗi con** lặp dài nhất (một đáp án bất kỳ): khôi phục từ DP.
2. **Lặp ít nhất `k` lần** (`k ≥ 2`): nhị phân theo `L` và đếm tần suất cửa sổ.
3. **Không chồng lấn** (hai lần xuất hiện phải cách nhau `≥ L`): DP với `min(f, i - j)`.

Cài đặt và đối chiếu vét cạn:

In [29]:
def lrs_string(s):
    n = len(s)
    best, end = 0, -1
    prev = [0] * (n + 1)
    for i in range(n):
        cur = [0] * (n + 1)
        for j in range(i):
            if s[i] == s[j]:
                cur[j + 1] = prev[j] + 1
                if cur[j + 1] > best:
                    best, end = cur[j + 1], i
        prev = cur
    return s[end - best + 1:end + 1] if best else ""


def longest_k_times(s, k):
    def ok(L):
        cnt = {}
        for i in range(len(s) - L + 1):
            t = s[i:i + L]
            cnt[t] = cnt.get(t, 0) + 1
            if cnt[t] >= k:
                return True
        return False
    lo, hi = 0, len(s)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


def non_overlapping(s):
    return bug_non_overlapping(s)                       # DP có min(f, i - j)


def brute_k_times(s, k):
    n = len(s)
    for L in range(n, 0, -1):
        for i in range(n - L + 1):
            if occurrences(s, s[i:i + L]) >= k:
                return L
    return 0


def brute_non_overlap(s):
    n = len(s)
    for L in range(n // 2, 0, -1):
        for i in range(n - L + 1):
            t = s[i:i + L]
            if any(s[j:j + L] == t for j in range(i + L, n - L + 1)):
                return L
    return 0


rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 12)
    s = "".join(rng.choice("ab" if rng.random() < 0.6 else "abc") for _ in range(n))
    t = lrs_string(s)
    assert len(t) == lrs_brute(s) and (t == "" or occurrences(s, t) >= 2)
    for k in (2, 3, 4):
        assert longest_k_times(s, k) == brute_k_times(s, k), (s, k)
    assert non_overlapping(s) == brute_non_overlap(s), s
print("trả về chuỗi con / lặp ít nhất k lần / không chồng lấn: khớp vét cạn")

trả về chuỗi con / lặp ít nhất k lần / không chồng lấn: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | đếm theo định nghĩa | `O(n⁴)` | `O(n)` | oracle |
| 2. Nhị phân + `set` | đơn điệu theo `L` | `O(n² log n)` | `O(n²)` | đơn giản |
| 3. DP hậu tố chung | `f[i][j]` | `O(n²)` | `O(n)` | **đáp án chính** (`n ≤ 2000`) |
| 4. Nhị phân + băm cuộn | cửa sổ `O(1)` | `O(n log n)` | `O(n)` | `n` rất lớn |
| 5. Mảng hậu tố + LCP | LCP kề nhau | `O(n² log n)` | `O(n²)` | nhiều truy vấn chuỗi con |

Mọi cách khớp vét cạn trên `11 469` chuỗi nhỏ (nhị phân `n ≤ 12`, `{a,b,c}` `n ≤ 7`).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().longestRepeatingSubstring("aaaa"), Solution().longestRepeatingSubstring("abab"), Solution().longestRepeatingSubstring("abcabcabc"), Solution().longestRepeatingSubstring("abcdab"))

3 2 6 2


### Bài 2 (tính tay)

`s = "abab"`. Lập bảng `f[i][j]` cho mọi `j < i` (chỉ những ô `s[i] == s[j]`). Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP hậu tố chung:

In [31]:
def lrs_todo(s):
    n = len(s)
    best = 0
    prev = [0] * (n + 1)
    for i in range(n):
        cur = [0] * (n + 1)
        for j in range(i):
            pass        # TODO 1: nếu s[i] == s[j] thì cur[j + 1] = prev[j] + 1
            pass        # TODO 2: cập nhật best
        prev = cur
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(lrs_todo(s) == w for s, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **nhị phân + tập chuỗi con**:

In [32]:
class MySolution:
    def longestRepeatingSubstring(self, s):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().longestRepeatingSubstring(s) == w for s, w in examples)

### Bài 5 (mini challenge): đếm số chuỗi con lặp dài nhất

Viết `count_longest(s)` trả về số **chuỗi con phân biệt** có độ dài bằng đáp án và xuất hiện `≥ 2` lần (`0` nếu đáp án `0`). Ví dụ `"abbaba"` → `2` (`"ab"`, `"ba"`).

In [33]:
def count_longest(s):
    pass            # TODO


# Khi làm xong:
# assert count_longest("abbaba") == 2 and count_longest("aabcaabdaab") == 1 and count_longest("abcd") == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `"aaaa"` lặp chồng lấn được.
- Mức 2: `"abcabcabc"`: `"abcabc"` ở `0` và `3`.
- Mức 3: `3 2 6 2`.

**Bài 2**

- Mức 1: các ô có `s[i] == s[j]`: `(2, 0)` (`a`), `(3, 1)` (`b`).
- Mức 2: `f[2][0] = 1`; `f[3][1] = f[2][0] + 1 = 2`.
- Mức 3: đáp án `2` (`"ab"` kết thúc tại `3` và `1`).

**Bài 3**

- Mức 1: `if s[i] == s[j]: cur[j + 1] = prev[j] + 1`.
- Mức 2: `best = max(best, cur[j + 1])` trong cùng nhánh.
- Mức 3: `prev = cur` sau mỗi `i`.

**Bài 4**

- Mức 1: `ok(L)`: dùng `set` các cửa sổ độ dài `L`.
- Mức 2: nhị phân lớn nhất `L` với `ok(L)`; `lo, hi = 0, n - 1`, `mid = (lo + hi + 1) // 2`.
- Mức 3: `if ok(mid): lo = mid else: hi = mid - 1`.

**Bài 5**

- Mức 1: tìm `L = Solution().longestRepeatingSubstring(s)`.
- Mức 2: đếm tần suất mọi cửa sổ độ dài `L` bằng `dict`.
- Mức 3: đếm số chuỗi có tần suất `≥ 2`; trả `0` nếu `L = 0`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().longestRepeatingSubstring("aaaa"), Solution().longestRepeatingSubstring("abab"), Solution().longestRepeatingSubstring("abcabcabc"), Solution().longestRepeatingSubstring("abcdab"))
print(vals)
assert vals == (3, 2, 6, 2) == tuple(lrs_brute(s) for s in ("aaaa", "abab", "abcabcabc", "abcdab"))
print("Bài 1 OK")

(3, 2, 6, 2)
Bài 1 OK


### Bài 2

In [35]:
assert trace_dp("abab") == 2 == lrs_brute("abab")
print("Bài 2 OK")

s = 'abab'
  i = 2 ('a'), j = 0: f = 1  (chuỗi con 'a')
  i = 3 ('b'), j = 1: f = 2  (chuỗi con 'ab')
  đáp án = 2
Bài 2 OK


### Bài 3

In [36]:
def lrs_answer(s):
    n = len(s)
    best = 0
    prev = [0] * (n + 1)
    for i in range(n):
        cur = [0] * (n + 1)
        for j in range(i):
            if s[i] == s[j]:
                cur[j + 1] = prev[j] + 1
                best = max(best, cur[j + 1])
        prev = cur
    return best


assert all(lrs_answer(s) == w for s, w in examples)
for s in pool:
    assert lrs_answer(s) == lrs_brute(s)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def longestRepeatingSubstring(self, s):
        def ok(L):
            seen = set()
            for i in range(len(s) - L + 1):
                t = s[i:i + L]
                if t in seen:
                    return True
                seen.add(t)
            return False

        lo, hi = 0, len(s) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo


assert all(MySolutionAnswer().longestRepeatingSubstring(s) == w for s, w in examples)
for s in pool:
    assert MySolutionAnswer().longestRepeatingSubstring(s) == lrs_brute(s)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def count_longest(s):
    L = Solution().longestRepeatingSubstring(s)
    if L == 0:
        return 0
    cnt = {}
    for i in range(len(s) - L + 1):
        t = s[i:i + L]
        cnt[t] = cnt.get(t, 0) + 1
    return sum(1 for v in cnt.values() if v >= 2)


def brute_count_longest(s):
    L = lrs_brute(s)
    if L == 0:
        return 0
    return len({s[i:i + L] for i in range(len(s) - L + 1) if occurrences(s, s[i:i + L]) >= 2})


assert count_longest("abbaba") == 2 and count_longest("aabcaabdaab") == 1 and count_longest("abcd") == 0
for s in pool:
    assert count_longest(s) == brute_count_longest(s)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chuỗi con lặp dài nhất, chồng lấn được phép, `n ≤ 2000`):**

```python
class Solution:
    def longestRepeatingSubstring(self, s: str) -> int:
        n = len(s)
        best = 0
        prev = [0] * (n + 1)
        for i in range(n):
            cur = [0] * (n + 1)
            for j in range(i):
                if s[i] == s[j]:
                    cur[j + 1] = prev[j] + 1
                    best = max(best, cur[j + 1])
            prev = cur
        return best
```

**Checklist trước khi submit:**

- [ ] `j < i` (hai vị trí khác nhau), không có `j == i`?
- [ ] Chồng lấn được phép (không cắt `f` bởi `i - j`)?
- [ ] Nhị phân: `lo = 0`, `hi = n - 1`, `mid = (lo + hi + 1) // 2`, đủ `n - L + 1` cửa sổ?
- [ ] Băm cuộn có kiểm chứng khi trùng (hoặc modulo lớn / hai modulo)?
- [ ] Đã thử `"a"` → `0`, `"aaaa"` → `3`, `"abcd"` → `0`, `"abbaba"` → `2`?
- [ ] Đúng tên `class Solution`, `longestRepeatingSubstring(self, s)`, không in debug.